In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from IPython.display import display, HTML
from sans_cdss import parameters as prm
from sans_cdss.generator import make_astronauts, make_sessions
from sans_cdss.triage import triage_sessions, explain
from sans_cdss import evaluate as ev
astronauts = make_astronauts(2000, seed=0)
sessions = make_sessions(astronauts, seed=1)
triaged = triage_sessions(sessions)

# SANS Triage Prototype: demo
2,000 synthetic astronauts · 12,504 in-flight exams · rule-based triage on NASA's published SANS thresholds. Synthetic data only, not clinical guidance.

## 1. Every number has a source label

In [2]:
prov = prm.provenance_table()
counts = prov.kind.value_counts().reindex(["reported", "derived", "assumption"]).rename("parameters").to_frame()
display(counts)
keep = ["p_gf_moderate", "p_distortion_given_macular", "p_vf_loss_if_dtrt100", "beyond_correction_D", "p_mtrr66_risk_allele"]
weak = prov.set_index("parameter").loc[keep, ["value", "note"]].reset_index()
weak["value"] = weak["value"].astype(str)
print("Least-supported assumptions (for clinician review):")
display(weak.style.hide(axis="index").set_properties(subset=["note"], **{"text-align": "left"}))

,parameters
kind,
reported,21
derived,11
assumption,23


Least-supported assumptions (for clinician review):


parameter,value,note
p_gf_moderate,0.3,not reported; placeholder
p_distortion_given_macular,0.15,not reported; placeholder
p_vf_loss_if_dtrt100,0.25,not reported; placeholder
beyond_correction_D,1.75,placeholder for 'beyond correction'
p_mtrr66_risk_allele,0.5,approx; verify in gnomAD


## 2. Each decision explains itself

In [3]:
for t in (1, 2, 3):
    row = triaged[triaged.tier == t].iloc[0]
    print(f"{row.astronaut_id}, flight day {row.flight_day}")
    import textwrap
    for line in explain(row).splitlines():
        print(textwrap.fill(line, 100, subsequent_indent="                "))
    print()

SYN-0003, flight day 30
Tier 1: Earliest indication
  because: dTRT 25 um >= 20 um
  draft action: Increase surveillance: add interim OCT; review trend at next exam; notify flight
                surgeon at routine conference.

SYN-0002, flight day 30
Tier 2: Clinically concerning
  because: Frisen grade >= 1 on fundus imaging
  draft action: Flight surgeon + ground ophthalmology review (timing TBD by clinicians); consider
                countermeasures per flight surgeon; flag for post-flight lumbar puncture (NASA:
                within 7-14 days of return).

SYN-0038, flight day 30
Tier 3: Acute functional impact
  because: Visual field loss (e.g., enlarged blind spot)
  draft action: Urgent escalation to flight surgeon; functional vision testing; mission-impact
                assessment.



## 3. Final tier across the cohort

In [4]:
display(ev.tier_summary(triaged).style.format({"share": "{:.0%}"}))

,n,share
0 - No SANS signs,359,18%
1 - Earliest indication,863,43%
2 - Clinically concerning,726,36%
3 - Acute functional impact,52,3%


## 4. What if the clinically concerning cut-off moved from 55 to 45 µm?

In [5]:
s = ev.threshold_sensitivity(sessions, earliest=(20,), concerning=(55, 45))
s = s.rename(columns={"concerning_um": "Tier 2 cut-off (µm)", "pct_tier2plus": "Tier 2 or higher",
                      "pct_reclassified": "Astronauts reclassified"}).drop(columns="earliest_um")
display(s.style.format({"Tier 2 or higher": "{:.0f}%", "Astronauts reclassified": "{:.0f}%"}).hide(axis="index"))

Tier 2 cut-off (µm),Astronauts reclassified,Tier 2 or higher
55,0%,39%
45,7%,46%
